In [ ]:
import pandas as pd

# load just the datasets q01 needs.
factor = 10
DATA_ROOT = f"/home/dias-benchmarks/tpch/data/factor_{factor}"
STORAGE_OPTS = {}  # or load from JSON

def load_lineitem(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/lineitem.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    print(df.columns)
    df["L_SHIPDATE"] = pd.to_datetime(df.L_SHIPDATE, format="%Y-%m-%d")
    df["L_RECEIPTDATE"] = pd.to_datetime(df.L_RECEIPTDATE, format="%Y-%m-%d")
    df["L_COMMITDATE"] = pd.to_datetime(df.L_COMMITDATE, format="%Y-%m-%d")
    return df
lineitem = load_lineitem(DATA_ROOT, **STORAGE_OPTS)

In [ ]:
def load_supplier(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/supplier.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    return df
supplier = load_supplier(DATA_ROOT, **STORAGE_OPTS)

In [ ]:
def load_orders(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/orders.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    df["O_ORDERDATE"] = pd.to_datetime(df.O_ORDERDATE, format="%Y-%m-%d")
    return df
orders = load_orders(DATA_ROOT, **STORAGE_OPTS)

In [ ]:
def load_customer(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/customer.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    return df
customer = load_customer(DATA_ROOT, **STORAGE_OPTS)

In [ ]:
def load_nation(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/nation.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    return df
nation = load_nation(DATA_ROOT, **STORAGE_OPTS)

In [ ]:
### cell 0 ###

# Optimize nation filtering by mapping
nation_map = nation.set_index('N_NATIONKEY')['N_NAME']

# Filter customers in France or Germany and map to CUST_NATION
cust_ng = (
    customer[['C_CUSTKEY', 'C_NATIONKEY']]
    .assign(CUST_NATION=lambda df: df['C_NATIONKEY'].map(nation_map))
    .query("CUST_NATION in ['FRANCE', 'GERMANY']")
    [['C_CUSTKEY', 'CUST_NATION']]
)

# Filter suppliers in France or Germany and map to SUPP_NATION
supp_ng = (
    supplier[['S_SUPPKEY', 'S_NATIONKEY']]
    .assign(SUPP_NATION=lambda df: df['S_NATIONKEY'].map(nation_map))
    .query("SUPP_NATION in ['FRANCE', 'GERMANY']")
    [['S_SUPPKEY', 'SUPP_NATION']]
)

# Filter lineitem by date, compute L_YEAR and VOLUME in one chained operation
li_f = (
    lineitem.loc[
        (lineitem['L_SHIPDATE'] >= pd.Timestamp('1995-01-01')) &
        (lineitem['L_SHIPDATE'] < pd.Timestamp('1997-01-01')),
        ['L_ORDERKEY', 'L_SUPPKEY', 'L_SHIPDATE', 'L_EXTENDEDPRICE', 'L_DISCOUNT']
    ]
    .assign(
        L_YEAR=lambda df: df['L_SHIPDATE'].dt.year,
        VOLUME=lambda df: df['L_EXTENDEDPRICE'] * (1.0 - df['L_DISCOUNT'])
    )
    [['L_ORDERKEY', 'L_SUPPKEY', 'L_YEAR', 'VOLUME']]
)


df = (
    li_f
    # attach supplier nation
    .merge(supp_ng, left_on='L_SUPPKEY', right_on='S_SUPPKEY', how='inner')
    # attach order -> customer key
    .merge(orders[['O_ORDERKEY', 'O_CUSTKEY']], left_on='L_ORDERKEY', right_on='O_ORDERKEY', how='inner')
    # attach customer nation
    .merge(cust_ng, left_on='O_CUSTKEY', right_on='C_CUSTKEY', how='inner')
    # only opposite-nation trade: France<->Germany
    .query('SUPP_NATION != CUST_NATION')
    # select final columns and aggregate
    [['SUPP_NATION', 'CUST_NATION', 'L_YEAR', 'VOLUME']]
)

In [ ]:
### cell 1 ###

total = (
    df
    .groupby(['SUPP_NATION', 'CUST_NATION', 'L_YEAR'], as_index=False)
    .agg(REVENUE=('VOLUME', 'sum'))
)